#Z-ordering

1. z-ordering is optimization technique in databricks for delta table.
2. it's reorganized data based on one or more column, improving data skipping , reducing data scanning and improve query performance.

3. Zordering helps co_locate related data in the same file.
4. when you are frequently filter or join on column, zordering arrange the data so those value are stored closer on 
   the disk.fewer files need to read.

In [0]:
Df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/hotal_catalog/hotel_schema/hotel_datafiles/hotels.csv")

display(Df)



###Cleaning Dataframe column

In [0]:
import re

for old_col in Df.columns:
  new_col = re.sub(r'[^a-zA-Z0-9_]', '_', old_col)
  new_col = new_col.strip('_').lower()
  Df = Df.withColumnRenamed(old_col, new_col)
  
display(Df)

####Remaove Duplicate

In [0]:
Df1 = Df.dropDuplicates()
display(Df1)


####Write Data in Delta Format

In [0]:
Df1.write.format('Delta').mode('Overwrite').save('/Volumes/hotal_catalog/hotel_schema/hotel_datafiles/hotels_zordering/')

###Display Path

In [0]:
path = dbutils.fs.ls("/Volumes/hotal_catalog/hotel_schema/hotel_datafiles/hotels_zordering/")
display(path)

####Optimize Data based on Z-Ordering column

In [0]:
spark.sql("""OPTIMIZE delta.`/Volumes/hotal_catalog/hotel_schema/hotel_datafiles/hotels_zordering/` ZORDER BY 
          (country_name)""")